In [2]:
from typing import List
from enum import Enum
import seaborn as sns
import os
import pandas as pd
import json
from statistics import mean
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [3]:
class DatasetNames(Enum):
    """Enum class for the dataset names"""

    KAR = "kar"
    WORDS = "words"
    VOTE = "vote"
    POW = "pow"
    FB_75 = "fb-75"
    COND_MAT = "cond-mat"
   
class DetectionAlgorithmsNames(Enum):
    """Enum class for the detection algorithms"""
    
    GRE = "greedy"
    LOUV = "louvain"
    WALK = "walktrap"
    LEID = "leiden"
    INF = "infomap"
    LAB = "label_propagation"
    EIG = "leading_eigenvector"
    BTW = "edge_betweenness"
    SPIN = "spinglass"
    SCD = "scd"
    LOC = "locale"
    DGC = "dgcluster"

class EvasionAlgorithmsNames(Enum):
    RAND = "random"
    DEG = "degree"
    BETW = "betweenness"
    ROAM = "roam"
    DICE = "dice"
    NABLA = "nabla-cmh"
    DRL = "drl-agent"
    GRE = "greedy"

def check_dir(path: str):
        """
        Check if the directory exists, if not create it.

        Parameters
        ----------
        path : str
            Path to the directory
        """
        if not os.path.exists(path):
            os.makedirs(path)

## Recompute plots of a run

In [6]:
def plot_metrics(
        datasets: List[str],
        evasion_algs: List[str],
        detection_algs: List[str],
        budget_factors: List[float],
        taus: List[float],
        metrics: List[str],
        seed: int,
    ) -> None:
        """
        Plot the metrics for the evasion algorithms.

        Parameters
        ----------
        datasets : List[str]
            List of datasets
        evasion_algs : List[str]
            List of evasion algorithms
        detection_algs : List[str]
            List of detection algorithms
        budget_factors : List[float]
            List of budget factors
        taus : List[float]
            List of tau values
        metrics : List[str]
            List of metrics
        output_dir : str
            Output directory
        """

        output_dir = f"../outputs_review/runs/seed_{seed}/"
        plots_dir = "/plots/"
        for dataset in datasets:
            dataset_name = getattr(DatasetNames, dataset).value
            for alg in detection_algs:
                alg_name = getattr(DetectionAlgorithmsNames, alg).value
                for tau in taus:
                    for c_beta in budget_factors:

                        # ----------------- Load the results ----------------- #
                        results_dir = output_dir + f"{dataset_name}/{alg_name}/tau_{tau}/betaFactor_{c_beta}/json_results/"
                        output_plots_dir = output_dir + f"{dataset_name}/{alg_name}/tau_{tau}/betaFactor_{c_beta}" + plots_dir
                        check_dir(output_plots_dir)
                        results = {}
                        for evasion_alg in evasion_algs:
                            evasion_alg_name = getattr(EvasionAlgorithmsNames, evasion_alg).value
                            file_name = results_dir + f"{evasion_alg_name}.json"
                            with open(file_name, "r", encoding="utf-8") as f:
                                log = json.load(f)
                            results[evasion_alg] = log
                        budget = max(results[evasion_algs[0]]["steps"]) # for steps plot

                        # ----------------- Store/compute metrics ----------------- #
                        metrics_data = {}
                        for metric in metrics:
                            if metric == "f1":
                                df = pd.DataFrame(
                                    {
                                        "Algorithm": evasion_algs,
                                        metric.capitalize(): [
                                            0 if (mean(results[alg]["goal"]) + mean(results[alg]["nmi"])) == 0 else 
                                            2 * mean(results[alg]["goal"]) * mean(results[alg]["nmi"]) / 
                                            (mean(results[alg]["goal"]) + mean(results[alg]["nmi"]))
                                            for alg in evasion_algs
                                        ],
                                    }
                                )
                            elif metric == "steps":
                                df = pd.DataFrame(
                                    {
                                        "Algorithm": evasion_algs,
                                        metric.capitalize(): [
                                            mean([results[alg][metric][i] for i in range(len(results[alg]["goal"])) if results[alg]["goal"][i] == 1])/budget 
                                            if any(results[alg]["goal"][i] == 1 for i in range(len(results[alg]["goal"]))) else 0 
                                            for alg in evasion_algs],
                                    }
                                )
                            else:
                                df = pd.DataFrame(
                                    {
                                        "Algorithm": evasion_algs,
                                        metric.capitalize(): [mean(results[alg][metric]) for alg in evasion_algs],
                                    }
                                )
                            # Convert the goal column to percentage
                            if metric == "goal":
                                df[metric.capitalize()] = df[metric.capitalize()] * 100
                            # Convert the budget column to percentage
                            if metric == "steps":
                                df[metric.capitalize()] = df[metric.capitalize()] * 100
                            
                            # Store metric data for JSON
                            metrics_data[metric] = df.set_index("Algorithm").to_dict()[metric.capitalize()]

                            # ----------------- Plot ----------------- #
                            if len(evasion_algs) > 1:
                                sns.barplot(
                                    data=df,
                                    x="Algorithm",
                                    y=metric.capitalize(),
                                    hue="Algorithm",
                                    palette=sns.color_palette("tab10", n_colors=len(evasion_algs)),
                                    edgecolor="black",  
                                    linewidth=0.5
                                )
                                plt.title(
                                    f"Evaluation on {dataset_name} graph with {alg_name} algorithm"
                                )
                                plt.xlabel("Algorithm")
                                if metric == "goal":
                                    plt.ylabel(f"{metric.capitalize()} reached %")
                                elif metric == "time":
                                    plt.ylabel(f"{metric.capitalize()} (s)")
                                elif metric == "steps":
                                    plt.ylabel("Budget used % if goal reached")
                                else:
                                    plt.ylabel(metric.capitalize())
                                file_path = output_plots_dir + f"{metric}.png"
                                plt.savefig(file_path)
                                plt.clf()

                        # Save metrics data to JSON
                        metrics_json_path = output_plots_dir + "metrics.json"
                        with open(metrics_json_path, "w", encoding="utf-8") as json_file:
                            json.dump(metrics_data, json_file, indent=4)

In [7]:
graph_names = ["FB_75"]

community_detection_algs = ["GRE", "LOUV", "WALK", "LEID", "INF", "LAB", "SCD", "LOC", "DGC"]

evasion_algs = ["RAND", "DEG", "BETW", "ROAM", "DICE", "NABLA", "DRL"] 

beta_factors = [2]

taus = [0.5]  

seed = 2025

metrics = ["goal","nmi","f1","time","steps"]

plot_metrics(
    datasets=graph_names,
    evasion_algs=evasion_algs,
    detection_algs=community_detection_algs,
    budget_factors=beta_factors,
    taus=taus,
    metrics=metrics, 
    seed =seed,
)

<Figure size 640x480 with 0 Axes>

## Replace only some methods

In [2]:
import json


def rewrite_json(input_path: str, output_path: str):
    """
    Reads a JSON file from the input path and writes it to the output path.

    Args:
        input_path (str): Path to the input JSON file.
        output_path (str): Path to the output JSON file.
    """
    try:
        # Read the JSON file
        with open(input_path, 'r') as infile:
            data = json.load(infile)
        
        # Write the JSON file to the new path

        with open(output_path, 'w') as outfile:
            json.dump(data, outfile, indent=4)
    except Exception as e:
        print(f"An error occurred: {e}")


algorithms = ["dgcluster", "leiden", "louvain", "infomap", "walktrap", "greedy", "label_propagation", "scd", "locale"]

alg_to_copy = ["nabla-cmh", "dice"]

betas = [2]

input_path = "../outputs/2025-05-10/13-21-25/fb-75"

output_path = "../outputs_review/runs/seed_2025/fb-75"

for detect in algorithms:
    for beta in betas:
        for alg in alg_to_copy:
            # Construct the input and output file paths
            input_file = f"{input_path}/{detect}/tau_0.5/betaFactor_{beta}/json_results/{alg}.json"
            output_file = f"{output_path}/{detect}/tau_0.5/betaFactor_{beta}/json_results/{alg}.json"
            
            # Rewrite the JSON file
            rewrite_json(input_file, output_file)

## Compose a complete run from singles runs on different betas

In [13]:
import json


def copy_folder(input_folder: str, output_folder: str):
    """
    Copies the contents of a folder from the input path to the output path.

    Args:
        input_folder (str): Path to the input folder.
        output_folder (str): Path to the output folder.
    """
    try:
        # Check if the output folder exists, if not create it
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
        
        # Iterate through all files in the input folder
        for root, _, files in os.walk(input_folder):
            for file in files:
                # Construct full file paths
                input_file_path = os.path.join(root, file)
                relative_path = os.path.relpath(root, input_folder)
                output_file_path = os.path.join(output_folder, relative_path, file)
                
                # Ensure the output directory exists
                output_dir = os.path.dirname(output_file_path)
                if not os.path.exists(output_dir):
                    os.makedirs(output_dir)
                
                # Copy the file
                with open(input_file_path, 'r') as infile:
                    data = json.load(infile)
                with open(output_file_path, 'w') as outfile:
                    json.dump(data, outfile, indent=4)
    except Exception as e:
        print(f"An error occurred: {e}")


algorithms = ["dgcluster", "leiden", "louvain", "infomap", "walktrap", "greedy", "label_propagation", "scd", "locale"]

input_paths = [
    "../outputs/2025-05-06/15-57-23/fb-75",
    "../outputs/2025-05-06/15-57-45/fb-75",
    "../outputs/2025-05-06/15-58-10/fb-75"
]

output_path = "../outputs_review/runs/seed_2025/fb-75"

betas = [0.5,1,2]

for input_path in input_paths:
    for detect in algorithms:
        for beta in betas:
            # Construct the input and output file paths
            input_file = f"{input_path}/{detect}/tau_0.5/betaFactor_{beta}/json_results/"
            output_file = f"{output_path}/{detect}/tau_0.5/betaFactor_{beta}/json_results/"
            check_dir(output_file)
            
            # Copy the folder contents
            copy_folder(input_file, output_file)
                

## Compose a complete run from singles runs on different taus

In [8]:
import os
import shutil


def copy_folder(input_folder: str, output_folder: str):
    """
    Copies the contents of a folder from the input path to the output path.

    Args:
        input_folder (str): Path to the input folder.
        output_folder (str): Path to the output folder.
    """
    try:
        # Check if the output folder exists, if not create it
        if os.path.exists(output_folder):
            shutil.rmtree(output_folder)
        
        # Copy the folder contents
        shutil.copytree(input_folder, output_folder)
    except Exception as e:
        print(f"An error occurred: {e}")


algorithms = ["dgcluster", "leiden", "louvain", "infomap", "walktrap", "greedy", "label_propagation", "scd", "locale"]

input_path = "../outputs/2025-05-11/21-10-12/vote"

output_path = "../outputs_review/runs/seed_42/vote"

taus = [0.3,0.8]

for detect in algorithms:
        for tau in taus:
            # Construct the input and output file paths
            input_file = f"{input_path}/{detect}/tau_{tau}/"
            output_file = f"{output_path}/{detect}/tau_{tau}/"
            check_dir(output_file)
            
            # Copy the folder contents
            copy_folder(input_file, output_file)
                